# Exploit Text Graph — End-to-End Demo

Runs **both research thrusts** from Samtani's NSF CAREER proposal (pages 9–14) in one notebook:

- **RT1**: build per-spell Exploit Text Graphs → train Diachronic Graph Transformer → detect semantic shifts → forecast with ARIMA → compare to word2vec + naive baselines.
- **RT2**: pretrain Context Transformer Encoder with masked + contrastive objectives → fine-tune on EV pairs → rank → compare to TF-IDF, BM25, LSA, and optional SimCSE / Contriever.

Default is `Config.tiny()` — full end-to-end run in ~2 minutes on MPS/CPU, useful for sanity-checking the pipeline. Swap to `Config.smoke()` for a richer but still-laptop-friendly run, or `Config.full()` for a cluster-scale run.

## 1. Setup

In [ ]:
from __future__ import annotations

import dataclasses
import os, sys, json, subprocess, warnings, importlib, time
warnings.filterwarnings('ignore')

# Ensure notebook runs from repo root (so caches land in ./cache).
# Detection uses pyproject.toml as a landmark — robust whether the
# notebook is opened in Jupyter (cwd = notebooks/) or executed via
# nbconvert (cwd = repo root).
from pathlib import Path as _Path
_cwd = _Path.cwd()
if (_cwd / 'pyproject.toml').exists():
    REPO = str(_cwd)            # already at repo root (nbconvert)
elif (_cwd.parent / 'pyproject.toml').exists():
    REPO = str(_cwd.parent)     # one level up (Jupyter from notebooks/)
else:
    REPO = str(_cwd)            # fallback
os.chdir(REPO)
print('working dir:', os.getcwd())

# Ensure etg package is importable.
# Uses sys.path injection (works in locked cluster venvs where pip install fails).
try:
    import etg  # noqa: F401
except ModuleNotFoundError:
    _src = os.path.join(REPO, 'src')
    if _src not in sys.path:
        sys.path.insert(0, _src)
        print(f'etg not installed — added {_src} to sys.path')
    import etg  # noqa: F401 — retry after path fix

from etg.config import Config
from etg.device import auto_device, device_summary
from etg.seeding import set_global_seed
from etg.logging_utils import get_logger

# ── Config selection ──────────────────────────────────────────────────────────
# Config.smoke()  — 2 000 posts, 5 spells, d=64, K=16, ~20–40 min on MPS/CPU.
#                   Good for comparing all models on a laptop.
# Config.tiny()   — 400 posts, 4 spells, d=32 — < 2 min, pipeline smoke-test only.
# Config.full()   — cluster-scale (500k posts, GPU required).
config = Config.smoke()

# ── Data source ───────────────────────────────────────────────────────────────
# "real"      → load CAREER-aligned RT1 corpora + real RT2 pairs from data/ev_pairs.jsonl
# "synthetic" → generate a fresh CTI-flavoured corpus from config
DATA_SOURCE = "real"
CAREER_RT1_SOURCES = None                 # None → use default_rt1_paths()
CAREER_RT2_SOURCE = os.path.join("data", "ev_pairs.jsonl")

# Set True to include SimCSE / Contriever baselines (downloads ~500 MB from HuggingFace).
RUN_PRETRAINED_BASELINES = False

config.ensure_dirs()
set_global_seed(config.seed)
device = auto_device()
log = get_logger('etg.notebook')
log.info(f'device = {device_summary(device)}')
log.info(f'config hash = {config.hash()}')
log.info(f'data_source = {DATA_SOURCE}')
log.info(f'n_posts={config.n_posts:,}  n_spells={config.n_spells}  '
         f'n_ev_pairs={config.n_ev_pairs:,}  vocab_cap={config.vocab_size_cap:,}')
log.info(f'DGT: dim={config.dgt_hidden_dim}  heads={config.dgt_num_heads}  '
         f'layers={config.dgt_num_layers}  epochs={config.dgt_epochs}')
log.info(f'CTE: emb={config.cte_emb_dim}  lstm={config.cte_lstm_hidden}  '
         f'pretrain_epochs={config.cte_pretrain_epochs}  ft_epochs={config.cte_finetune_epochs}')
_T0 = time.time()

results: dict = {'config_hash': config.hash(), 'rt1': {}, 'rt2': {}}


## 2. Data

### 2a. Data source selection

`DATA_SOURCE` (set in Setup cell above) controls which corpus feeds the pipeline:

| Value | Source | Notes |
|-------|--------|-------|
| `"real"` | Curated RT1 community corpora + `data/ev_pairs.jsonl` | Uses `load_career_rt1_posts()` and `load_career_rt2_pairs()` so the notebook matches `python -m etg.career_run`. |
| `"synthetic"` | Generative CTI-flavoured corpus | Seeded, reproducible, injects known semantic shifts for ground-truth evaluation. |

**Real-data loading stages** (run automatically when `DATA_SOURCE == "real"`):
1. Resolve the default RT1 source files via `default_rt1_paths()` unless `CAREER_RT1_SOURCES` is overridden.
2. Clean and filter each `ForumPost` with the same lightweight language/length checks used by the CAREER runner.
3. Rank posts by CTI signal and keep a balanced sample across `config.n_spells` time-spells.
4. Load a deterministic RT2 subset from `data/ev_pairs.jsonl` via `load_career_rt2_pairs()`.
5. Preserve real exploit-matched negatives for RT2 when downsampling so the smoke run stays aligned with the award framing.

The broader raw-crawl preprocessing pipeline under `etg.data.preprocessing` is still available for separate experiments, but it is no longer the default notebook path.


In [ ]:
# ── 2b. Load CAREER-aligned real data (real data only) ───────────────────────
if DATA_SOURCE == "real":
    from collections import Counter as _Ctr
    from etg.data.career_award import default_rt1_paths, load_career_rt1_posts
    from etg.data.time_spells import TimeSpellIndex as _TSI

    _rt1_paths = (
        [str(p) for p in default_rt1_paths()]
        if CAREER_RT1_SOURCES is None
        else [str(p) for p in CAREER_RT1_SOURCES]
    )

    print("Loading CAREER-aligned RT1 corpora …")
    for _path in _rt1_paths:
        print(" ", _path)

    _t = __import__('time').time()
    _real_posts = load_career_rt1_posts(config, paths=_rt1_paths)
    print(f"Load wall-time: {__import__('time').time() - _t:.1f}s")
    print()

    _tsi = _TSI.from_posts(_real_posts, config.n_spells)
    _spell_dist = _Ctr(_tsi.assign(p.timestamp) for p in _real_posts)
    _ts_sorted = sorted(p.timestamp for p in _real_posts)
    _forum_dist = _Ctr(p.forum_id for p in _real_posts)

    print("Posts per spell:", dict(sorted(_spell_dist.items())))
    print(f"Date range: {_ts_sorted[0].strftime('%Y-%m-%d')} → {_ts_sorted[-1].strftime('%Y-%m-%d')}")
    print(f"Forum distribution: {dict(_forum_dist.most_common(10))}")

    _rt1_meta = {
        "sources": _rt1_paths,
        "forums": len(_forum_dist),
        "date_range": (_ts_sorted[0], _ts_sorted[-1]),
    }
else:
    print("Using synthetic data — skipping CAREER real-data loaders.")
    _real_posts = None
    _rt1_meta = None


In [ ]:
# ── Table S2. CAREER RT1 spell balance ───────────────────────────────────────
import pandas as pd

if DATA_SOURCE == "real" and _real_posts is not None and _rt1_meta is not None:
    from collections import Counter as _Ctr
    from etg.data.time_spells import TimeSpellIndex as _TSI

    _tsi = _TSI.from_posts(_real_posts, config.n_spells)
    _spell_counts = _Ctr(_tsi.assign(p.timestamp) for p in _real_posts)
    _spell_df = pd.DataFrame(
        [{"Spell": spell, "Posts": _spell_counts.get(spell, 0)} for spell in range(config.n_spells)]
    ).set_index("Spell")
    _spell_df["Pct"] = (_spell_df["Posts"] / max(len(_real_posts), 1) * 100).round(1).astype(str) + "%"

    print("Table S2. CAREER RT1 Spell Balance")
    print("─" * 50)
    print(_spell_df.to_string())
    print("─" * 50)
    print(f"Sources: {len(_rt1_meta['sources'])} files  Forums: {_rt1_meta['forums']}")
else:
    print("(CAREER RT1 spell summary not available — using synthetic data.)")


In [ ]:
if DATA_SOURCE == "real" and _real_posts is not None:
    from etg.data.career_award import load_career_rt2_pairs

    posts = _real_posts
    ev_pairs = load_career_rt2_pairs(config, path=CAREER_RT2_SOURCE)
    gt_shift_words: set[str] = set()   # no injected shifts in real data
    print(f"[real]  {len(posts):,} posts loaded from curated RT1 corpora")
    print(f"[real]  RT2 source: {CAREER_RT2_SOURCE}")
else:
    from etg.data.synthetic_ev import SyntheticEV
    from etg.data.synthetic_forum import SyntheticForum

    _syn_loader = SyntheticForum(config)
    posts = list(_syn_loader)
    gt_shift_words = _syn_loader.ground_truth_shift_words()
    print(f"[synthetic]  {len(posts):,} posts generated")
    print(f"  ground-truth shift words: {sorted(gt_shift_words)}")

    ev_loader = SyntheticEV(config)
    ev_pairs = list(ev_loader)

print(f"EV pairs:       {len(ev_pairs):>6,} ({sum(p.label==1 for p in ev_pairs):,} positive)")
print()
print("sample post:")
print(" ", posts[0].text[:160])
print()
print("sample EV pair:")
print("  E:", ev_pairs[0].exploit_text[:120])
print("  V:", ev_pairs[0].vulnerability_text[:120])


In [ ]:
# ── Table S1. Corpus composition by forum source ─────────────────────────────
import pandas as pd
import numpy as np
from collections import defaultdict

_CTI_KWS_SUMM = [
    "exploit", "cve", "rce", "lpe", "payload", "malware", "ransomware",
    "backdoor", "c2", "shellcode", "0day", "zeroday", "poc",
    "overflow", "injection", "privilege", "escalation", "pentest",
]

def _quick_cti(text: str) -> int:
    t = text.lower()
    return sum(t.count(k) for k in _CTI_KWS_SUMM)

_by_forum: dict = defaultdict(list)
for _p in posts:
    _by_forum[_p.forum_id].append(_p)

_src_rows = []
for _fid, _fps in sorted(_by_forum.items(), key=lambda x: -len(x[1])):
    _dates   = sorted(p.timestamp for p in _fps)
    _lengths = [len(p.text.split()) for p in _fps]
    _scores  = [_quick_cti(p.text) for p in _fps]
    _src_rows.append({
        "Forum":             _fid,
        "Posts":             len(_fps),
        "Date range":        f"{_dates[0].strftime('%Y-%m')} \u2013 {_dates[-1].strftime('%Y-%m')}",
        "Med. tokens":       int(np.median(_lengths)),
        "Med. CTI score":    round(float(np.median(_scores)), 1),
        "CTI hit rate":      f"{sum(s > 0 for s in _scores) / len(_scores) * 100:.0f}%",
    })

# Totals
_all_lengths = [len(p.text.split()) for p in posts]
_all_scores  = [_quick_cti(p.text) for p in posts]
_all_dates   = sorted(p.timestamp for p in posts)
_src_rows.append({
    "Forum":          "\u2500\u2500 TOTAL \u2500\u2500",
    "Posts":          len(posts),
    "Date range":     f"{_all_dates[0].strftime('%Y-%m')} \u2013 {_all_dates[-1].strftime('%Y-%m')}",
    "Med. tokens":    int(np.median(_all_lengths)),
    "Med. CTI score": round(float(np.median(_all_scores)), 1),
    "CTI hit rate":   f"{sum(s > 0 for s in _all_scores) / len(_all_scores) * 100:.0f}%",
})

_src_df = pd.DataFrame(_src_rows).set_index("Forum")
print("Table S1. Corpus Composition by Forum Source  (after preprocessing)")
print("\u2500" * 80)
print(_src_df.to_string())
print("\u2500" * 80)
print(f"  Unique forums: {len(_by_forum)}   Unique authors: "
      f"{len(set(p.author_hash for p in posts)):,}   Total posts: {len(posts):,}")


## 3. RT1 — Diachronic Linguistics for Exploit Trend Detection

### 3a. Build per-spell Exploit Text Graphs (ETGs)

In [3]:
from etg.data.time_spells import TimeSpellIndex
from etg.rt1.vocab import build_vocab
from etg.rt1.etg_builder import build_etg_sequence, group_posts_by_spell
from etg.rt1.laplacian_pe import attach_lap_pe

vocab = build_vocab(posts, config)
time_spells = TimeSpellIndex.from_posts(posts, config.n_spells)
posts_by_spell = group_posts_by_spell(posts, time_spells)
snapshots = build_etg_sequence(posts, posts_by_spell, vocab, config)
attach_lap_pe(snapshots, config)

print(f'vocab size:        {vocab.size:>6,}')
print(f'spells:            {config.n_spells}')
for s in snapshots:
    print(f"  t={s['t']}: nodes_observed={int(s['node_mask'].sum())}, edges={s['edge_index'].shape[1]:,}")

vocab size:           196
spells:            5
  t=0: nodes_observed=130, edges=8,974
  t=1: nodes_observed=140, edges=11,069
  t=2: nodes_observed=140, edges=12,389
  t=3: nodes_observed=136, edges=13,127
  t=4: nodes_observed=141, edges=13,570


In [ ]:
# ── Figure S1. Vocabulary and edge-count growth across time spells ────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os

_spell_labels = [f"T{s['t'] + 1}" for s in snapshots]
_vocab_sizes  = [int(s["node_mask"].sum()) for s in snapshots]
_edge_counts  = [s["edge_index"].shape[1] for s in snapshots]

fig, ax1 = plt.subplots(figsize=(7, 3.5))
ax2 = ax1.twinx()

_bar_w = 0.6
ax1.bar(_spell_labels, _vocab_sizes, width=_bar_w, color="#4e79a7", alpha=0.75, label="Active vocab (nodes)")
ax2.plot(_spell_labels, _edge_counts, "o-", color="#e15759", linewidth=2.2, markersize=7, label="ETG edges")

ax1.set_xlabel("Time spell")
ax1.set_ylabel("Active vocabulary size", color="#4e79a7")
ax2.set_ylabel("ETG directed edges", color="#e15759")
ax1.tick_params(axis="y", labelcolor="#4e79a7")
ax2.tick_params(axis="y", labelcolor="#e15759")
for spine in ("top",):
    ax1.spines[spine].set_visible(False)
    ax2.spines[spine].set_visible(False)

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=8, framealpha=0.8)
ax1.set_title("Vocabulary & Graph Growth Across Time Spells\n"
              "(motivates diachronic modelling \u2014 the ETG changes substantially each spell)",
              fontsize=9, pad=6)

plt.tight_layout()
os.makedirs(str(config.figure_dir), exist_ok=True)
fig.savefig(str(config.figure_dir) + "/vocab_growth.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"  Saved: {config.figure_dir}/vocab_growth.png")
print(f"  Vocab growth:  T1={_vocab_sizes[0]:,} \u2192 T{len(_vocab_sizes)}={_vocab_sizes[-1]:,}  "
      f"({(_vocab_sizes[-1]-_vocab_sizes[0])/_vocab_sizes[0]*100:+.1f}%)")
print(f"  Edge growth:   T1={_edge_counts[0]:,} \u2192 T{len(_edge_counts)}={_edge_counts[-1]:,}  "
      f"({(_edge_counts[-1]-_edge_counts[0])/_edge_counts[0]*100:+.1f}%)")


### 3b. Visualize ETG degree distributions (Fig. 1)

In [4]:
import matplotlib.pyplot as plt
from etg.viz.rt1_plots import plot_etg_degree_distributions

fig = plot_etg_degree_distributions(
    snapshots, save_path=f'{config.figure_dir}/rt1_degree_distributions.png'
)
plt.show(); plt.close(fig)

### 3c. Train the Diachronic Graph Transformer

In [5]:
from etg.rt1.dgt_model import DGT
from etg.rt1.train_dgt import extract_embeddings, train_dgt

feature_dim = snapshots[0]['x'].shape[1]
dgt = DGT(config, feature_dim).to(device)
print(f'DGT parameters: {sum(p.numel() for p in dgt.parameters()):,}')

dgt_history = train_dgt(dgt, snapshots, config, device)
dgt_embeddings = extract_embeddings(dgt, snapshots, device)
print('embedding shapes:', [tuple(e.shape) for e in dgt_embeddings])

DGT parameters: 91,201


[04/15/26 07:09:48] INFO     DGT epoch 1/6 loss=1.5573 recon=1.1399 temp=0.8348

                    INFO     DGT epoch 2/6 loss=1.0270 recon=0.7014 temp=0.6512

                    INFO     DGT epoch 3/6 loss=0.8226 recon=0.5474 temp=0.5505

                    INFO     DGT epoch 4/6 loss=0.6876 recon=0.4478 temp=0.4794

                    INFO     DGT epoch 5/6 loss=0.6453 recon=0.4411 temp=0.4084

                    INFO     DGT epoch 6/6 loss=0.5578 recon=0.3920 temp=0.3314

embedding shapes: [(196, 64), (196, 64), (196, 64), (196, 64), (196, 64)]


### 3d. Detect semantic shifts and forecast with ARIMA

In [6]:
import numpy as np
from etg.rt1.shift_detection import detect_shifts_pairwise
from etg.rt1.eval_rt1 import extrinsic_forecast, intrinsic_shift_recall_at_k, mean_shift_per_pair, analogy_hit_rate
from etg.rt1.forecast_arima import rolling_forecast_eval

masks = [s['node_mask'] for s in snapshots]
shift_results = detect_shifts_pairwise(dgt_embeddings, masks, top_quantile=config.shift_top_quantile)
for r in shift_results:
    top_idx = np.argsort(-r.scores)[:10] if len(r.scores) else []
    top_words = [vocab.token(int(r.word_ids[i])) for i in top_idx]
    print(f't={r.t_prev}→{r.t_curr}  thr={r.threshold:.3f}  shifted={len(r.shifted_ids)}  top10={top_words}')

dgt_metrics = extrinsic_forecast(shift_results, config)

# shift_recall@k requires injected ground-truth words (synthetic data only)
if gt_shift_words:
    dgt_metrics['shift_recall@k'] = intrinsic_shift_recall_at_k(
        shift_results, vocab, gt_shift_words, k=config.analogy_hit_at_k * 10)
    dgt_metrics['analogy_hit@k']  = analogy_hit_rate(dgt_embeddings, vocab, k=config.analogy_hit_at_k)
else:
    print("(no ground-truth shift words — shift_recall@k and analogy_hit@k skipped for real data)")

results['rt1']['DGT'] = dgt_metrics
dgt_metrics


t=0→1  thr=0.015  shifted=6  top10=['see', 'postgres', 'the', 'works', 'dump', 'tested', 'attached', 'ssrf', 'day', 'first']
t=1→2  thr=0.009  shifted=7  top10=['spoof', 'v1', 'for', 'ship', 'to', 'tested', 'ready', 'csrf', 'works', 'nginx']
t=2→3  thr=0.010  shifted=7  top10=['chain', 'drop', 'spoof', 'new', 'see', 'via', 'csrf', 'shell', 'for', 'the']
t=3→4  thr=0.012  shifted=6  top10=['attached', 'need', 'see', 'poc', 'new', 'chain', 'the', 'ssrf', 'first', 'csrf']


{'MAE': 0.0005243306068925232,
 'RMSE': 0.0005243306068925232,
 'MAPE': 13.750218948234666,
 'R2': 0.0,
 'shift_recall@k': 0.4166666666666667,
 'analogy_hit@k': 1.0}

In [ ]:
# ── Table 3. Top semantically-shifted terms per spell pair (DGT) ─────────────
import pandas as pd
import numpy as np

_shift_rows = []
for _r in shift_results:
    if len(_r.scores) == 0:
        continue
    _order = np.argsort(-_r.scores)
    _top_n = min(10, len(_order))
    for _rank, _oi in enumerate(_order[:_top_n], 1):
        _word    = vocab.token(int(_r.word_ids[_oi]))
        _score   = float(_r.scores[_oi])
        _shifted = int(_r.word_ids[_oi]) in set(_r.shifted_ids.tolist() if hasattr(_r.shifted_ids, "tolist") else _r.shifted_ids)
        _gt_mark = "\u2605" if _word in (gt_shift_words or set()) else ""
        _shift_rows.append({
            "Spell pair":  f"T{_r.t_prev + 1}\u2192T{_r.t_curr + 1}",
            "Rank":        _rank,
            "Term":        _word + _gt_mark,
            "Shift score": round(_score, 4),
            "Above thr.":  "\u2713" if _shifted else "",
        })

if _shift_rows:
    _shift_df = pd.DataFrame(_shift_rows)
    print("Table 3. Top-10 Semantically Shifted Terms per Time-Spell Pair (DGT embeddings)")
    print("  Shift score = 1 \u2212 cosine(z_v^t, z_v^{t\u22121}).  \u2605 = injected ground-truth (synthetic only).")
    print("\u2500" * 70)
    for _sp, _grp in _shift_df.groupby("Spell pair"):
        print(f"\n  {_sp}  (threshold = {[r for r in shift_results if f'T{r.t_prev+1}\u2192T{r.t_curr+1}' == _sp][0].threshold:.4f})")
        print(_grp[["Rank", "Term", "Shift score", "Above thr."]].to_string(index=False))
    print("\n" + "\u2500" * 70)
    if gt_shift_words:
        _detected_in_top5 = set(_shift_df[_shift_df["Rank"] <= 5]["Term"].str.replace("\u2605",""))
        _found = _detected_in_top5 & gt_shift_words
        print(f"  Ground-truth shifts: {sorted(gt_shift_words)}")
        print(f"  Detected in top-5:   {sorted(_found)}  ({len(_found)}/{len(gt_shift_words)} recall)")
else:
    print("  No shift results to display.")


In [ ]:
# ── Figure S2. Word embedding drift trajectories over time ───────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import os

# Collect per-word mean shift across spell pairs
_word_mean_shift: dict = {}
for _r in shift_results:
    for _i, _wid in enumerate(_r.word_ids):
        _w = vocab.token(int(_wid))
        _word_mean_shift.setdefault(_w, []).append(float(_r.scores[_i]))

_mean_shifts = {w: float(np.mean(v)) for w, v in _word_mean_shift.items() if len(v) >= max(1, len(snapshots) - 2)}

if len(_mean_shifts) >= 4:
    _sorted_ws = sorted(_mean_shifts.items(), key=lambda x: -x[1])
    # Top 4 drifting + 2 most stable (present across all spells)
    _track = [w for w, _ in _sorted_ws[:4]] + [w for w, _ in _sorted_ws[-2:]]
    _track = list(dict.fromkeys(_track))[:6]

    _PALETTE = ["#e15759", "#4e79a7", "#f28e2b", "#76b7b2", "#59a14f", "#b07aa1"]
    _T = len(dgt_embeddings)
    fig, ax = plt.subplots(figsize=(8, 4))

    for _ci, _term in enumerate(_track):
        _idx = vocab.tokens_to_id.get(_term)
        if _idx is None:
            continue
        _z0 = dgt_embeddings[0][_idx]
        _dists = []
        for _t in range(_T):
            _zt = dgt_embeddings[_t][_idx]
            _n0, _nt = float(np.linalg.norm(_z0)), float(np.linalg.norm(_zt))
            if _n0 > 1e-8 and _nt > 1e-8:
                _dists.append(1.0 - float(np.dot(_z0, _zt) / (_n0 * _nt)))
            else:
                _dists.append(0.0)

        _is_high = _mean_shifts.get(_term, 0) > float(np.median(list(_mean_shifts.values())))
        _style   = "-o" if _is_high else "--s"
        _alpha   = 0.9 if _is_high else 0.6
        ax.plot([f"T{i+1}" for i in range(_T)], _dists,
                _style, color=_PALETTE[_ci % len(_PALETTE)],
                linewidth=2, markersize=6, label=_term, alpha=_alpha)

    ax.axhline(0, color="gray", linewidth=0.7, linestyle=":")
    ax.set_xlabel("Time spell")
    ax.set_ylabel("Cosine drift from T1 embedding")
    ax.set_title("DGT Word Embedding Drift Over Time\n"
                 "Solid/circles = high-drift terms  \u00b7  Dashed/squares = stable terms",
                 fontsize=9.5, pad=6)
    ax.legend(fontsize=8, ncol=2, loc="upper left", framealpha=0.8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    fig.savefig(str(config.figure_dir) + "/word_trajectories.png", dpi=300, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {config.figure_dir}/word_trajectories.png")
    print(f"  Tracked terms: {_track}")
else:
    print("  Not enough tracked terms to plot trajectories.")


In [ ]:
# ── Table S3. Per-spell ETG statistics ───────────────────────────────────────
import pandas as pd
import numpy as np

_etg_rows = []
for _snap in snapshots:
    _t       = _snap["t"]
    _V_act   = int(_snap["node_mask"].sum())
    _E       = _snap["edge_index"].shape[1]
    _density = _E / max(_V_act * (_V_act - 1), 1) * 1000  # \u00d710\u207b\u00b3

    # Weighted out-degree \u2192 top hub nodes
    _src = _snap["edge_index"][0]
    _wt  = _snap["edge_weight"]
    _src_np = _src.numpy() if hasattr(_src, "numpy") else np.asarray(_src)
    _wt_np  = _wt.numpy()  if hasattr(_wt,  "numpy") else np.asarray(_wt)
    _deg = np.zeros(vocab.size, dtype=np.float32)
    np.add.at(_deg, _src_np, _wt_np)
    _top5_ids   = np.argsort(_deg)[::-1][:5]
    _top5_words = [vocab.token(int(i)) for i in _top5_ids if _deg[i] > 0]

    # New / lost nodes vs previous spell
    if _t > 0:
        _prev_mask = snapshots[_t - 1]["node_mask"]
        _curr_mask = _snap["node_mask"]
        _new_nodes  = int((~_prev_mask & _curr_mask).sum())
        _lost_nodes = int((_prev_mask & ~_curr_mask).sum())
        _churn = f"+{_new_nodes}/\u2212{_lost_nodes}"
    else:
        _churn = "\u2014"

    _etg_rows.append({
        "Spell":             f"T{_t + 1}",
        "Active nodes":      _V_act,
        "Edges":             _E,
        "Density (\u00d710\u207b\u00b3)":  round(_density, 3),
        "Node churn":        _churn,
        "Top-5 hub nodes":   ", ".join(_top5_words[:5]),
    })

_etg_df = pd.DataFrame(_etg_rows).set_index("Spell")
print("Table S3. ETG Statistics per Time Spell")
print("\u2500" * 88)
print(_etg_df.to_string())
print("\u2500" * 88)
print(f"  Mean active nodes: {_etg_df['Active nodes'].mean():.0f}  "
      f"  Mean edges: {_etg_df['Edges'].mean():.0f}  "
      f"  Mean density: {_etg_df['Density (\u00d710\u207b\u00b3)'].mean():.4f}\u00d710\u207b\u00b3")


### 3e. Visualize shifts, forecast, and word trajectories (Figs. 2–5)

In [7]:
from etg.viz.rt1_plots import (
    plot_shift_score_histograms,
    plot_forecast_band,
    plot_embedding_trajectories_umap,
    plot_attention_heatmap,
)

fig = plot_shift_score_histograms(shift_results, save_path=f'{config.figure_dir}/rt1_shift_hist.png')
plt.show(); plt.close(fig)

series = mean_shift_per_pair(shift_results)
preds, actuals = rolling_forecast_eval(series, min_train=max(2, min(3, len(series)-1)), order=config.arima_order)
fig = plot_forecast_band(series, preds, actuals, save_path=f'{config.figure_dir}/rt1_forecast.png')
plt.show(); plt.close(fig)

watch = sorted(gt_shift_words)[:6]
fig = plot_embedding_trajectories_umap(dgt_embeddings, vocab, watch, save_path=f'{config.figure_dir}/rt1_trajectories.png')
plt.show(); plt.close(fig)

# Attention heatmap on a small subgraph for interpretation.
from etg.rt1.dgt_model import _build_adjacency_mask
import torch
sample_words = [w for w in ('sqli','rce','apache','nginx','exploit','payload','creds','token') if w in vocab.tokens_to_id][:6]
sample_ids = [vocab.id(w) for w in sample_words]
with torch.no_grad():
    snap = snapshots[-1]
    x = snap['x'].to(device); pe = snap['lap_pe'].to(device)
    h0 = dgt.input_proj(torch.cat([x, pe], dim=-1))
    V = x.shape[0]
    adj = _build_adjacency_mask(snap['edge_index'].to(device), V, config.dgt_k_hop, device)
    sub = h0[sample_ids]
    scores = (sub @ sub.T / (sub.shape[-1] ** 0.5))
    attn = torch.softmax(scores, dim=-1).cpu().numpy()
fig = plot_attention_heatmap(attn, sample_words, save_path=f'{config.figure_dir}/rt1_attention_heatmap.png')
plt.show(); plt.close(fig)

### 3f. RT1 baselines: word2vec (aligned) and static GCN

In [8]:
from etg.eval.metrics_forecast import all_metrics
from etg.rt1.forecast_arima import rolling_forecast_eval, forecast_series
from etg.baselines.rt1_baselines import (
    # classical / statistical
    bow_cosine_drift_series, freq_shift_series, cusum_series,
    ppmi_svd_per_spell, pmi_ratio_series, lda_topic_drift_series,
    # word-vector
    train_word2vec_per_spell, word2vec_shift_series,
    train_fasttext_per_spell, train_glove_per_spell,
    # random-walk graph embedding
    train_deepwalk_per_spell,
    # neural graph (static)
    train_line_per_spell, train_gcn_per_spell, train_gat_per_spell, train_gae_per_spell,
    train_graphsage_per_spell, train_gin_per_spell,
    # neural graph (temporal)
    train_tgat_per_spell, train_evolvegcn_per_spell,
    # structural PE
    lapPE_direct_series,
    # forecasting baselines
    ets_forecast_series, random_shift_series,
    # helpers
    embeddings_to_shift_series,
)

# ── shared evaluation helper ─────────────────────────────────────────────────
def _rt1_eval(series, label):
    preds, actuals = rolling_forecast_eval(
        series, min_train=max(2, min(3, len(series) - 1)), order=config.arima_order
    )
    if len(actuals):
        results['rt1'][label] = all_metrics(np.asarray(actuals), np.asarray(preds))
    else:
        results['rt1'][label] = {m: float('nan') for m in ('MAE', 'RMSE', 'MAPE', 'R2')}
    mae  = results['rt1'][label]['MAE']
    rmse = results['rt1'][label]['RMSE']
    mape = results['rt1'][label]['MAPE']
    print(f'  {label:32s}  {mae:>10.5f}  {rmse:>10.5f}  {mape:>8.2f}')

print('\u2500' * 72)
print(f"  {'Model':32s}  {'MAE':>10}  {'RMSE':>10}  {'MAPE':>8}")
print('\u2500' * 72)

# ── 0. Random (absolute floor) ────────────────────────────────────────────────
_rt1_eval(random_shift_series(shift_results),                      'Random')

# ── 1. Naive last-value ───────────────────────────────────────────────────────
series = mean_shift_per_pair(shift_results)
naive_preds   = [forecast_series(series[:k], horizon=1, min_points=1)[0] for k in range(2, len(series))]
naive_actuals = list(series[2:])
if naive_actuals:
    results['rt1']['naive_last'] = all_metrics(np.asarray(naive_actuals), np.asarray(naive_preds))
else:
    results['rt1']['naive_last'] = {m: float('nan') for m in ('MAE', 'RMSE', 'MAPE', 'R2')}
print(f"  {'naive_last':32s}  {results['rt1']['naive_last']['MAE']:>10.5f}  "
      f"{results['rt1']['naive_last']['RMSE']:>10.5f}  {results['rt1']['naive_last']['MAPE']:>8.2f}")

# ── 2. ETS / Holt-Winters ─────────────────────────────────────────────────────
ets_preds, ets_actuals = ets_forecast_series(series)
if ets_actuals:
    results['rt1']['ETS'] = all_metrics(np.asarray(ets_actuals), np.asarray(ets_preds))
else:
    results['rt1']['ETS'] = {m: float('nan') for m in ('MAE', 'RMSE', 'MAPE', 'R2')}
print(f"  {'ETS':32s}  {results['rt1']['ETS']['MAE']:>10.5f}  "
      f"{results['rt1']['ETS']['RMSE']:>10.5f}  {results['rt1']['ETS']['MAPE']:>8.2f}")

# ── 3-7. Statistical baselines ────────────────────────────────────────────────
_rt1_eval(bow_cosine_drift_series(posts_by_spell, config),         'BoW-drift')
_rt1_eval(freq_shift_series(snapshots),                            'FreqShift')
_rt1_eval(cusum_series(series),                                    'CUSUM-DGT')
_rt1_eval(pmi_ratio_series(snapshots, vocab),                      'PMI-ratio')
_rt1_eval(lda_topic_drift_series(posts_by_spell, config),          'LDA-drift')

# ── 8-9. Matrix factorization ─────────────────────────────────────────────────
_rt1_eval(embeddings_to_shift_series(ppmi_svd_per_spell(snapshots, vocab, config)), 'PPMI-SVD')
_rt1_eval(lapPE_direct_series(snapshots, config),                  'LapPE-direct')

# ── 10-12. Word vectors ───────────────────────────────────────────────────────
w2v_emb = train_word2vec_per_spell(posts_by_spell, vocab, config)
_rt1_eval(word2vec_shift_series(w2v_emb),                          'word2vec')
_rt1_eval(embeddings_to_shift_series(train_fasttext_per_spell(posts_by_spell, vocab, config)), 'fasttext')
_rt1_eval(embeddings_to_shift_series(train_glove_per_spell(posts_by_spell, vocab, config)),    'GloVe')

# ── 13. Random-walk graph embeddings (Node2Vec removed — too slow on real ETGs) ─
_rt1_eval(embeddings_to_shift_series(train_deepwalk_per_spell(snapshots, vocab, config)), 'DeepWalk')

# ── 14-19. Static neural graph embeddings ────────────────────────────────────
_rt1_eval(embeddings_to_shift_series(train_line_per_spell(snapshots, vocab, config, device)),        'LINE')
_rt1_eval(embeddings_to_shift_series(train_gcn_per_spell(snapshots, vocab, config, device)),         'StaticGCN')
_rt1_eval(embeddings_to_shift_series(train_gat_per_spell(snapshots, vocab, config, device)),         'GAT')
_rt1_eval(embeddings_to_shift_series(train_gae_per_spell(snapshots, vocab, config, device)),         'GAE')
_rt1_eval(embeddings_to_shift_series(train_graphsage_per_spell(snapshots, vocab, config, device)),   'GraphSAGE')
_rt1_eval(embeddings_to_shift_series(train_gin_per_spell(snapshots, vocab, config, device)),         'GIN')

# ── 20-21. Temporal graph embeddings ─────────────────────────────────────────
_rt1_eval(embeddings_to_shift_series(train_tgat_per_spell(snapshots, vocab, config, device)),        'TGAT')
_rt1_eval(embeddings_to_shift_series(train_evolvegcn_per_spell(snapshots, vocab, config, device)),   'EvolveGCN')

print('\u2500' * 72)
print()
results['rt1']

── RT1 baselines ──────────────────────────────────────────────────────
  Random                            MAE=0.13275  RMSE=0.13275  MAPE=24.36
  naive_last                        MAE=0.00035
  ETS                               MAE=0.00035
  BoW-drift                         MAE=0.00441  RMSE=0.00441  MAPE=10.84
  FreqShift                         MAE=0.01368  RMSE=0.01368  MAPE=5.79
  CUSUM-DGT                         MAE=0.00000  RMSE=0.00000  MAPE=0.00


  PMI-ratio                         MAE=0.24197  RMSE=0.24197  MAPE=10.06


  LDA-drift                         MAE=0.02352  RMSE=0.02352  MAPE=18.95


  PPMI-SVD                          MAE=0.06207  RMSE=0.06207  MAPE=27.36
  LapPE-direct                      MAE=0.00336  RMSE=0.00336  MAPE=2.21


  word2vec                          MAE=0.00308  RMSE=0.00308  MAPE=105.37


...

  GraphSAGE                         MAE=0.02350  RMSE=0.02350  MAPE=94.22


  GIN                               MAE=0.03009  RMSE=0.03009  MAPE=56.66


  TGAT                              MAE=0.00334  RMSE=0.00334  MAPE=5.76


  EvolveGCN                         MAE=0.00011  RMSE=0.00011  MAPE=9.44



{'DGT': {'MAE': 0.0005243306068925232,
  'RMSE': 0.0005243306068925232,
  'MAPE': 13.750218948234666,
  'R2': 0.0,
  'shift_recall@k': 0.4166666666666667,
  'analogy_hit@k': 1.0},
 'Random': {'MAE': 0.13274778168093604,
  'RMSE': 0.13274778168093604,
  'MAPE': 24.362613092739192,
  'R2': 0.0},
 'naive_last': {'MAE': 0.00034722706872215717,
  'RMSE': 0.0003897849413277069,
  'MAPE': 9.461414930534277,
  'R2': -1.21060910563192},
 'ETS': {'MAE': 0.0003472233656793833,
  'RMSE': 0.0003897799600748338,
  'MAPE': 9.461317820717976,
  'R2': -1.2105526050777429},
 'BoW-drift': {'MAE': 0.00440737284125315,
  'RMSE': 0.00440737284125315,
  'MAPE': 10.844564499747424,
  'R2': 0.0},
 'FreqShift': {'MAE': 0.013680827150026886,
  'RMSE': 0.013680827150026886,
  'MAPE': 5.785659042244052,
  'R2': 0.0},
 'CUSUM-DGT': {'MAE': 0.0, 'RMSE': 0.0, 'MAPE': 0.0, 'R2': 0.0},
 'PMI-ratio': {'MAE': 0.2419730159087421,
  'RMSE': 0.2419730159087421,
  'MAPE': 10.061990037532606,
  'R2': 0.0},
 'LDA-drift': {'MAE

### 3g. RT1 summary bar chart

In [9]:
from etg.viz.rt1_plots import plot_rt1_summary_bar
fig = plot_rt1_summary_bar(results['rt1'], metric='MAE', save_path=f'{config.figure_dir}/rt1_summary_mae.png')
plt.show(); plt.close(fig)
fig = plot_rt1_summary_bar(results['rt1'], metric='RMSE', save_path=f'{config.figure_dir}/rt1_summary_rmse.png')
plt.show(); plt.close(fig)

## 4. RT2 — Exploit-Vulnerability Self-Supervised Linker

### 4a. CTE pretraining (masked + contrastive)

In [10]:
from etg.rt2.cte_model import CTE
from etg.rt2.pretrain_cte import pretrain_cte

cte = CTE(config).to(device)
print(f'CTE parameters: {sum(p.numel() for p in cte.parameters()):,}')
pre_hist = pretrain_cte(cte, ev_pairs, config, device)
print('final pretrain losses:', {k: round(v[-1], 4) for k, v in pre_hist.items()})

CTE parameters: 2,307,648


[04/15/26 07:10:35] INFO     CTE pretrain epoch 1/4 loss=5.4050 mlm=4.3748 cont=1.0302

[04/15/26 07:10:49] INFO     CTE pretrain epoch 2/4 loss=1.9286 mlm=1.5304 cont=0.3982

[04/15/26 07:11:06] INFO     CTE pretrain epoch 3/4 loss=1.5311 mlm=1.2139 cont=0.3172

[04/15/26 07:11:24] INFO     CTE pretrain epoch 4/4 loss=1.4053 mlm=1.1268 cont=0.2785

final pretrain losses: {'loss': 1.4053, 'mlm': 1.1268, 'contrastive': 0.2785}


### 4b. CTE supervised fine-tuning

In [11]:
from etg.rt2.finetune_cte import finetune_cte
ft_hist = finetune_cte(cte, ev_pairs, config, device)
print('final fine-tune loss:', round(ft_hist['loss'][-1], 4))

[04/15/26 07:11:25] INFO     CTE fine-tune epoch 1/4 loss=0.0225

[04/15/26 07:11:26] INFO     CTE fine-tune epoch 2/4 loss=0.0049

[04/15/26 07:11:27] INFO     CTE fine-tune epoch 3/4 loss=0.0033

                    INFO     CTE fine-tune epoch 4/4 loss=0.0030

final fine-tune loss: 0.003


### 4c. Rank exploits and compute HR/MRR/NDCG/MAP

In [12]:
from etg.rt2.ranking import rank_exploits, sample_candidate_pool
from etg.rt2.eval_rt2 import evaluate_ranking

positive_pairs = [p for p in ev_pairs if p.label == 1]
pool = sample_candidate_pool(ev_pairs, n_candidates=max(100, len(positive_pairs)//2), seed=config.seed)
scores_cte, pos_idx = rank_exploits(cte, positive_pairs, pool, config, device)
results['rt2']['CTE'] = evaluate_ranking(scores_cte, pos_idx, k=config.eval_top_k)
results['rt2']['CTE']

{'HR@10': 0.9333333333333333,
 'MRR@10': 0.411936507936508,
 'NDCG@10': 0.5358685575414087,
 'MAP': 0.41602378809316093,
 'median_rank': 3.0,
 'mean_rank': 4.7725}

### 4d. RT2 baselines (TF-IDF / BM25 / LSA / SimCSE / Contriever)

In [13]:
from etg.baselines.rt2_baselines import (
    build_tfidf_ranker,
    tfidf_score, lsa_score,
    bm25_score_matrix, bm25_plus_score_matrix, bm25_l_score_matrix,
    query_likelihood_score,
    jaccard_score_matrix, char_ngram_score_matrix,
    rouge_l_score_matrix, edit_distance_score_matrix, wmd_score_matrix,
    logistic_score_matrix, svm_score_matrix, random_forest_score_matrix,
    xgboost_score_matrix, hand_feature_gb_score_matrix,
    esim_score_matrix, match_pyramid_score_matrix, dssm_score_matrix,
    w2v_mean_score_matrix, fasttext_mean_score_matrix, doc2vec_score_matrix,
    glove_mean_score_matrix, autoencoder_score_matrix, vae_score_matrix,
    siamese_bilstm_score_matrix, siamese_bilstm_hardneg_score_matrix,
    cnn_encoder_score_matrix, bilstm_cnn_score_matrix, knrm_score_matrix,
    cte_mlm_only_score_matrix, cte_contrastive_only_score_matrix,
    cte_finetune_only_score_matrix, cte_no_temporal_score_matrix,
    cte_shared_encoder_score_matrix, cte_temperature_score_matrix,
    simcse_score, contriever_score,
)
from etg.eval.metrics_ir import ranks_from_scores, all_ranking_metrics
from etg.rt2.eval_rt2 import evaluate_ranking

exploit_texts  = [p.exploit_text for p in positive_pairs]
seen: dict[str, int] = {}
for t in [p.vulnerability_text for p in positive_pairs]:
    seen.setdefault(t, len(seen))
for t in pool:
    seen.setdefault(t, len(seen))
candidate_texts  = list(seen.keys())
positive_indices = np.array([seen[p.vulnerability_text] for p in positive_pairs])
k_eval           = config.eval_top_k
vec = build_tfidf_ranker(ev_pairs)

# per-query NDCG storage for significance testing
_rt2_per_query: dict[str, list] = {}

def _rt2_eval(score_matrix, name, *, store_per_query=False):
    # Robust call: handle older eval_rt2.py that lacks return_per_query
    try:
        m, pq = evaluate_ranking(score_matrix, positive_indices, config.eval_top_k, return_per_query=True)
    except TypeError:
        m = evaluate_ranking(score_matrix, positive_indices, config.eval_top_k)
        try:
            from etg.eval.significance import per_query_ndcg as _pqn
            pq = _pqn(score_matrix, positive_indices, k=config.eval_top_k)
        except Exception:
            pq = None
    results['rt2'][name] = m
    if store_per_query and pq is not None:
        _rt2_per_query[name] = pq
    _pct = f"  HR@10={m['HR@10']:.3f}  MRR={m['MRR@10']:.3f}  NDCG={m['NDCG@10']:.3f}  MAP={m['MAP']:.3f}"
    print(f'  {name:36s}{_pct}')

print('\u2500\u2500 RT2 baselines \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')

# Sparse retrieval
_rt2_eval(tfidf_score(vec, exploit_texts, candidate_texts),                  'TF-IDF', store_per_query=True)
_rt2_eval(lsa_score(vec, exploit_texts, candidate_texts, dim=min(64, max(2, len(candidate_texts)-1))), 'LSA')
_rt2_eval(bm25_score_matrix(exploit_texts, candidate_texts),                 'BM25', store_per_query=True)
_rt2_eval(bm25_plus_score_matrix(exploit_texts, candidate_texts),            'BM25+')
_rt2_eval(bm25_l_score_matrix(exploit_texts, candidate_texts),               'BM25L')
_rt2_eval(query_likelihood_score(exploit_texts, candidate_texts),            'QueryLikelihood')

# Lexical overlap
_rt2_eval(jaccard_score_matrix(exploit_texts, candidate_texts),              'Jaccard')
_rt2_eval(char_ngram_score_matrix(exploit_texts, candidate_texts),           'CharNgram(3,4)')
_rt2_eval(rouge_l_score_matrix(exploit_texts, candidate_texts),              'ROUGE-L')
_rt2_eval(edit_distance_score_matrix(exploit_texts, candidate_texts),        'EditDist')
_rt2_eval(wmd_score_matrix(ev_pairs, exploit_texts, candidate_texts, config),'WMD')

# Supervised ML
_rt2_eval(logistic_score_matrix(vec, exploit_texts, candidate_texts, positive_indices),      'LogisticReg')
_rt2_eval(svm_score_matrix(vec, exploit_texts, candidate_texts, positive_indices),           'LinearSVM')
_rt2_eval(random_forest_score_matrix(vec, exploit_texts, candidate_texts, positive_indices), 'RandomForest')
_rt2_eval(xgboost_score_matrix(vec, exploit_texts, candidate_texts, positive_indices),       'XGBoost(GB)')
_rt2_eval(hand_feature_gb_score_matrix(exploit_texts, candidate_texts, positive_indices),    'HandFeature-GB')

# Interaction-based
_rt2_eval(dssm_score_matrix(ev_pairs, exploit_texts, candidate_texts, config, device),          'DSSM')
_rt2_eval(esim_score_matrix(ev_pairs, exploit_texts, candidate_texts, config, device),          'ESIM-lite')
_rt2_eval(match_pyramid_score_matrix(ev_pairs, exploit_texts, candidate_texts, config, device), 'MatchPyramid')

# Neural encoders
_rt2_eval(glove_mean_score_matrix(ev_pairs, exploit_texts, candidate_texts, config),             'GloVe-Mean')
_rt2_eval(w2v_mean_score_matrix(ev_pairs, exploit_texts, candidate_texts, config),               'W2V-Mean')
_rt2_eval(fasttext_mean_score_matrix(ev_pairs, exploit_texts, candidate_texts, config),          'FastText-Mean')
_rt2_eval(doc2vec_score_matrix(ev_pairs, exploit_texts, candidate_texts, config),                'Doc2Vec')
_rt2_eval(autoencoder_score_matrix(ev_pairs, exploit_texts, candidate_texts, config, device),    'Autoencoder')
_rt2_eval(vae_score_matrix(ev_pairs, exploit_texts, candidate_texts, config, device),            'VAE')
_rt2_eval(knrm_score_matrix(ev_pairs, exploit_texts, candidate_texts, config, device),           'KNRM')
_rt2_eval(siamese_bilstm_score_matrix(ev_pairs, exploit_texts, candidate_texts, config, device),         'SiameseBiLSTM', store_per_query=True)
_rt2_eval(siamese_bilstm_hardneg_score_matrix(ev_pairs, exploit_texts, candidate_texts, config, device), 'SiameseBiLSTM-HardNeg', store_per_query=True)
_rt2_eval(cnn_encoder_score_matrix(ev_pairs, exploit_texts, candidate_texts, config, device),            'CNNEncoder')
_rt2_eval(bilstm_cnn_score_matrix(ev_pairs, exploit_texts, candidate_texts, config, device),             'BiLSTM-CNN')

# CTE ablations
print('  [CTE ablations]')
_rt2_eval(cte_mlm_only_score_matrix(ev_pairs, exploit_texts, candidate_texts, config, device),          'CTE-MLM-only')
_rt2_eval(cte_contrastive_only_score_matrix(ev_pairs, exploit_texts, candidate_texts, config, device),  'CTE-Contrastive-only')
_rt2_eval(cte_finetune_only_score_matrix(ev_pairs, exploit_texts, candidate_texts, config, device),     'CTE-FT-only')
_rt2_eval(cte_no_temporal_score_matrix(ev_pairs, exploit_texts, candidate_texts, config, device),       'CTE-no-temporal')
_rt2_eval(cte_shared_encoder_score_matrix(ev_pairs, exploit_texts, candidate_texts, config, device),    'CTE-dual-encoder', store_per_query=True)
for tau in (0.01, 0.07, 0.2):
    _rt2_eval(cte_temperature_score_matrix(ev_pairs, exploit_texts, candidate_texts, config, device, tau=tau), f'CTE-tau={tau}')

if RUN_PRETRAINED_BASELINES:
    for bname, fn in [('SimCSE', simcse_score), ('Contriever', contriever_score)]:
        try:
            s = fn(exploit_texts, candidate_texts)
        except Exception as ex:
            log.warning(f'{bname} unavailable: {ex}'); s = None
        if s is None:
            results['rt2'][bname] = {f'HR@{k_eval}': float('nan'), f'MRR@{k_eval}': float('nan'),
                                     f'NDCG@{k_eval}': float('nan'), 'MAP': float('nan')}
        else:
            _rt2_eval(s, bname)
else:
    log.info('Skipping SimCSE/Contriever (set RUN_PRETRAINED_BASELINES=True)')

# store per-query arrays in results for significance testing in summary cell
results['rt2_per_query'] = {k: v.tolist() for k, v in _rt2_per_query.items()}

print()
results['rt2']

── RT2 baselines ──────────────────────────────────────────────────────
  TF-IDF                            HR@10=0.842  MRR=0.315  NDCG=0.439  MAP=0.325


  LSA                               HR@10=0.848  MRR=0.301  NDCG=0.430  MAP=0.310


  BM25                              HR@10=0.849  MRR=0.311  NDCG=0.437  MAP=0.320


  BM25+                             HR@10=0.822  MRR=0.296  NDCG=0.419  MAP=0.307


  BM25L                             HR@10=0.828  MRR=0.299  NDCG=0.423  MAP=0.309


...

                    INFO     CTE fine-tune epoch 3/4 loss=0.0038

[04/15/26 07:31:13] INFO     CTE fine-tune epoch 4/4 loss=0.0030

  CTE-tau=0.2                       HR@10=0.776  MRR=0.310  NDCG=0.420  MAP=0.323


[04/15/26 07:31:13] INFO     Skipping SimCSE/Contriever (set RUN_PRETRAINED_BASELINES=True)

{'CTE': {'HR@10': 0.9333333333333333,
  'MRR@10': 0.411936507936508,
  'NDCG@10': 0.5358685575414087,
  'MAP': 0.41602378809316093,
  'median_rank': 3.0,
  'mean_rank': 4.7725},
 'TF-IDF': {'HR@10': 0.8416666666666667,
  'MRR@10': 0.31504100529100526,
  'NDCG@10': 0.43899579093112834,
  'MAP': 0.3248332055524264},
 'LSA': {'HR@10': 0.8483333333333334,
  'MRR@10': 0.30139484126984123,
  'NDCG@10': 0.42953503133683457,
  'MAP': 0.31039513262438256},
 'BM25': {'HR@10': 0.8491666666666666,
  'MRR@10': 0.31119477513227506,
  'NDCG@10': 0.43735565920947966,
  'MAP': 0.3202625623765621},
 'BM25+': {'HR@10': 0.8216666666666667,
  'MRR@10': 0.29613921957671957,
  'NDCG@10': 0.4189913934630163,
  'MAP': 0.30702801543570907},
 'BM25L': {'HR@10': 0.8275,
  'MRR@10': 0.29902347883597885,
  'NDCG@10': 0.4226302523268306,
  'MAP': 0.3094682983383469},
 'QueryLikelihood': {'HR@10': 0.6858333333333333,
  'MRR@10': 0.2604143518518519,
  'NDCG@10': 0.3608587122780331,
  'MAP': 0.27073304935673825},
 'Jac

In [ ]:
# ── Table 4. RT2 ranking case study: CTE vs TF-IDF ───────────────────────────
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

# Reconstruct the EXACT candidate list that rank_exploits() used internally:
#   positives first (unique), then pool distractors — so pos_idx aligns correctly.
_cs_exploit_texts  = [p.exploit_text        for p in positive_pairs]
_cs_positive_texts = [p.vulnerability_text  for p in positive_pairs]

_seen: dict = {}
for _t in _cs_positive_texts:
    _seen.setdefault(_t, len(_seen))
for _t in pool:
    _seen.setdefault(_t, len(_seen))
_cs_candidates = list(_seen.keys())   # same order as rank_exploits uses

# Build TF-IDF on the same (Q, C) space as scores_cte
_cs_tfidf = TfidfVectorizer(max_features=30000, sublinear_tf=True, ngram_range=(1, 2))
_cs_tfidf.fit(_cs_exploit_texts + _cs_candidates)
_cs_exp_mat  = _cs_tfidf.transform(_cs_exploit_texts)
_cs_cand_mat = _cs_tfidf.transform(_cs_candidates)
_tfidf_cs    = (_cs_exp_mat @ _cs_cand_mat.T).toarray()  # (Q, C)

# CTE scores from cell 27 — already aligned to the same candidate list
_cte_cs = scores_cte   # (Q, C)
_pos_cs = pos_idx      # (Q,)  — indices into _cs_candidates

def _safe_rank(score_row, pos_i):
    """Return 1-based rank of pos_i in score_row (descending). Fallback: len+1."""
    order = np.argsort(score_row)[::-1]
    hits  = np.where(order == pos_i)[0]
    return int(hits[0]) + 1 if len(hits) else len(score_row) + 1

_cte_ranks   = np.array([_safe_rank(_cte_cs[i],   _pos_cs[i]) for i in range(len(_pos_cs))])
_tfidf_ranks = np.array([_safe_rank(_tfidf_cs[i], _pos_cs[i]) for i in range(len(_pos_cs))])

# Pick 3 representative queries: best / median / worst by CTE rank
_by_rank   = np.argsort(_cte_ranks)
_pick_idxs = [int(_by_rank[0]), int(_by_rank[len(_by_rank) // 2]), int(_by_rank[-1])]
_pick_lbls = ["Best-case   (CTE rank 1)", "Median-case", "Worst-case  (hardest query)"]

print("Table 4. RT2 Ranking Case Study: CTE vs. TF-IDF")
print(f"  Pool size: {len(_cs_candidates)} candidates  |  Queries: {len(_pos_cs)}")
print("=" * 90)

for _qi, _lbl in zip(_pick_idxs, _pick_lbls):
    _exp  = _cs_exploit_texts[_qi]
    _gt   = _cs_candidates[_pos_cs[_qi]]
    _cr   = _cte_ranks[_qi]
    _tr   = _tfidf_ranks[_qi]
    _verd = "CTE better" if _cr < _tr else ("TF-IDF better" if _tr < _cr else "TIE")

    print(f"\n  [{_lbl}]")
    print(f"  Exploit  : {_exp[:130]}")
    print(f"  GT match : {_gt[:100]}")
    print(f"  CTE rank : {_cr:>4d}   TF-IDF rank : {_tr:>4d}   → {_verd}")
    print()

    _cte_top   = np.argsort(_cte_cs[_qi])[::-1][:5]
    _tfidf_top = np.argsort(_tfidf_cs[_qi])[::-1][:5]
    print(f"  {'k':>2}  {'CTE top-5':54s}  {'TF-IDF top-5':54s}")
    print(f"  {'──':>2}  {'─'*54}  {'─'*54}")
    for _k in range(min(5, len(_cs_candidates))):
        _ct  = _cs_candidates[_cte_top[_k]][:52]  + ".."
        _tt  = _cs_candidates[_tfidf_top[_k]][:52] + ".."
        _cgt = " ◄GT" if _cte_top[_k]   == _pos_cs[_qi] else "    "
        _tgt = " ◄GT" if _tfidf_top[_k] == _pos_cs[_qi] else "    "
        print(f"  {_k+1:>2}  {_ct:54s}{_cgt}  {_tt:54s}{_tgt}")

print("\n" + "=" * 90)
print(f"\n  Summary over all {len(_cte_ranks)} queries:")
print(f"    CTE    — median rank: {int(np.median(_cte_ranks)):>4d}  mean rank: {float(np.mean(_cte_ranks)):>6.1f}")
print(f"    TF-IDF — median rank: {int(np.median(_tfidf_ranks)):>4d}  mean rank: {float(np.mean(_tfidf_ranks)):>6.1f}")


In [ ]:
# ── Figure S3. Positive vs. negative pair score distributions (RT2) ──────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import os
from sklearn.feature_extraction.text import TfidfVectorizer as _TVec

_rng_dist = np.random.default_rng(42)

def _pos_neg_scores(score_matrix, pos_indices, n_neg=8):
    """Split score matrix into positive-pair scores and sampled-negative scores."""
    Q, C = score_matrix.shape
    pos_s = score_matrix[np.arange(Q), pos_indices]
    neg_s = []
    for _i in range(Q):
        _neg_pool = [_j for _j in range(C) if _j != pos_indices[_i]]
        if not _neg_pool:
            continue
        _sample = _rng_dist.choice(_neg_pool, size=min(n_neg, len(_neg_pool)), replace=False)
        neg_s.extend(score_matrix[_i, _sample].tolist())
    return pos_s, np.array(neg_s)

# Rebuild TF-IDF on the same candidate list rank_exploits used
_d_exploit_texts  = [p.exploit_text for p in positive_pairs]
_d_positive_texts = [p.vulnerability_text for p in positive_pairs]
_d_seen: dict = {}
for _t in _d_positive_texts:
    _d_seen.setdefault(_t, len(_d_seen))
for _t in pool:
    _d_seen.setdefault(_t, len(_d_seen))
_d_candidates = list(_d_seen.keys())

_d_tv  = _TVec(max_features=30000, sublinear_tf=True)
_d_tv.fit(_d_exploit_texts + _d_candidates)
_d_tfidf_mat = (_d_tv.transform(_d_exploit_texts) @ _d_tv.transform(_d_candidates).T).toarray()

_panels = [
    (_d_tfidf_mat, pos_idx, "TF-IDF"),
    (scores_cte,   pos_idx, "CTE (proposed)"),
]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for _ax, (_mat, _pi, _lbl) in zip(axes, _panels):
    _ps, _ns = _pos_neg_scores(_mat, _pi, n_neg=8)
    _ax.hist(_ns, bins=40, alpha=0.55, color="#4e79a7", density=True, label="Negative pairs")
    _ax.hist(_ps, bins=40, alpha=0.80, color="#e15759", density=True, label="Positive pairs")
    _delta = float(np.mean(_ps) - np.mean(_ns))
    _ax.set_title(f"{_lbl}\n(Δmean = {_delta:+.3f})", fontsize=10)
    _ax.set_xlabel("Similarity score")
    _ax.set_ylabel("Density")
    _ax.legend(fontsize=8)
    _ax.spines["top"].set_visible(False)
    _ax.spines["right"].set_visible(False)

fig.suptitle(
    "RT2 — Positive vs. Negative Pair Score Distributions\n"
    "Larger separation → better discriminability", fontsize=10.5)
plt.tight_layout()
os.makedirs(str(config.figure_dir), exist_ok=True)
fig.savefig(str(config.figure_dir) + "/score_distributions.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"  Saved: {config.figure_dir}/score_distributions.png")


In [ ]:
# ── Table 5. CTE ablation study: per-component contribution ──────────────────
import pandas as pd
import numpy as np

_ABL_DESCS = {
    "CTE-MLM-only":          "Removes InfoNCE contrastive + fine-tune",
    "CTE-Contrastive-only":  "Removes masked prediction + fine-tune",
    "CTE-FT-only":           "Removes both pretraining objectives",
    "CTE-no-temporal":       "Removes time-aware negative sampling",
    "CTE-tau=0.01":          "Temperature \u03c4 = 0.01 (very sharp)",
    "CTE-tau=0.07":          "Temperature \u03c4 = 0.07 (default)",
    "CTE-tau=0.2":           "Temperature \u03c4 = 0.20 (soft)",
    "CTE-dual-encoder":      "Shared-weight dual encoder variant",
}

_full_key = "CTE" if "CTE" in results["rt2"] else "CTE-dual-encoder"
_full      = results["rt2"].get(_full_key, {})

def _delta(new, ref, metric):
    r = ref.get(metric, float("nan"))
    n = new.get(metric, float("nan"))
    if r != r or n != n or r == 0:
        return float("nan")
    return (n - r) / abs(r) * 100

_abl_rows = []
for _name, _desc in _ABL_DESCS.items():
    _m = results["rt2"].get(_name)
    if _m is None:
        continue
    _abl_rows.append({
        "Model":          _name,
        "What changes":   _desc,
        "HR@10":          round(_m.get("HR@10",   float("nan")), 4),
        "NDCG@10":        round(_m.get("NDCG@10", float("nan")), 4),
        "MAP":            round(_m.get("MAP",      float("nan")), 4),
        "\u0394HR@10 (%)": f'{_delta(_m, _full, "HR@10"):+.1f}' if _full else "\u2014",
        "\u0394NDCG@10 (%)": f'{_delta(_m, _full, "NDCG@10"):+.1f}' if _full else "\u2014",
    })

# Reference row: full CTE
_abl_rows.append({
    "Model":         f"\u25c9 {_full_key} (full)",
    "What changes":  "\u2014 reference \u2014",
    "HR@10":         round(_full.get("HR@10",   float("nan")), 4),
    "NDCG@10":       round(_full.get("NDCG@10", float("nan")), 4),
    "MAP":           round(_full.get("MAP",     float("nan")), 4),
    "\u0394HR@10 (%)": "0.0 (ref)",
    "\u0394NDCG@10 (%)": "0.0 (ref)",
})

_abl_df = pd.DataFrame(_abl_rows).set_index("Model")
print("Table 5. CTE Ablation Study \u2014 Incremental Component Contribution")
print("  \u0394 = ablation metric \u2212 full-CTE metric  (negative = ablation is worse)")
print("\u2500" * 90)
print(_abl_df.to_string())
print("\u2500" * 90)
_best_abl = max(
    [r for r in _abl_rows if "ref" not in r["What changes"]],
    key=lambda x: x["HR@10"] if x["HR@10"] == x["HR@10"] else -1,
    default=None,
)
if _best_abl:
    print(f"  Strongest ablation: {_best_abl['Model']}  (HR@10={_best_abl['HR@10']:.4f})")


### 4e. Visualize CTE behavior (Figs. 6–10)

In [14]:
from etg.viz.rt2_plots import (
    plot_contrastive_view_umap,
    plot_score_histogram,
    plot_pr_roc,
    plot_cte_attention_heatmap,
    plot_rt2_summary_bar,
)
from etg.rt2.augmentations import contrastive_views
from etg.rt2.trigram_hasher import tokenize_cte
import random, torch

# (6) contrastive view UMAP
rng = random.Random(0)
sample_texts = [p.exploit_text for p in positive_pairs[:150]]
a_ids = torch.tensor([tokenize_cte(t, config.trigram_buckets, config.cte_max_len) for t in sample_texts], device=device)
b_ids = torch.tensor([tokenize_cte(contrastive_views(t, rng), config.trigram_buckets, config.cte_max_len) for t in sample_texts], device=device)
with torch.no_grad():
    z_a = cte.forward_contrastive(a_ids).cpu().numpy()
    z_b = cte.forward_contrastive(b_ids).cpu().numpy()
fig = plot_contrastive_view_umap(z_a, z_b, save_path=f'{config.figure_dir}/rt2_contrastive_umap.png')
plt.show(); plt.close(fig)

# (7) score histogram
fig = plot_score_histogram(scores_cte, pos_idx, save_path=f'{config.figure_dir}/rt2_score_hist.png')
plt.show(); plt.close(fig)

# (8) PR / ROC
fig = plot_pr_roc(scores_cte, pos_idx, save_path=f'{config.figure_dir}/rt2_pr_roc.png')
plt.show(); plt.close(fig)

# (9) CTE local-guided global attention on a sample exploit
import re
sample_exp = positive_pairs[0].exploit_text
toks = re.findall(r'[A-Za-z][A-Za-z0-9_\-]+', sample_exp)
with torch.no_grad():
    ids = torch.tensor([tokenize_cte(sample_exp, config.trigram_buckets, config.cte_max_len)], device=device)
    refined, sent, mask = cte.encode_tokens(ids)
    # Recompute the attention weights used by LG-GCA for display.
    pooled = (refined * mask.float().unsqueeze(-1)).sum(dim=1) / mask.float().sum(dim=1).clamp(min=1).unsqueeze(-1)
    Wh_g = pooled @ cte.lg_attn.W
    scores_local = torch.einsum('bld,bd->bl', refined, Wh_g) / (refined.shape[-1] ** 0.5)
    scores_local = scores_local.masked_fill(~mask, float('-inf'))
    attn_local = torch.softmax(scores_local, dim=-1).cpu().numpy()[0]
label_tokens = ['<CLS>'] + toks
label_tokens = label_tokens[: mask.sum().item()]
attn_local = attn_local[: len(label_tokens)]
fig = plot_cte_attention_heatmap(attn_local, label_tokens, save_path=f'{config.figure_dir}/rt2_cte_attention.png')
plt.show(); plt.close(fig)

# (10) RT2 summary bar
fig = plot_rt2_summary_bar(results['rt2'], metric=f'HR@{config.eval_top_k}', save_path=f'{config.figure_dir}/rt2_summary_hr.png')
plt.show(); plt.close(fig)
fig = plot_rt2_summary_bar(results['rt2'], metric=f'NDCG@{config.eval_top_k}', save_path=f'{config.figure_dir}/rt2_summary_ndcg.png')
plt.show(); plt.close(fig)

## 5. Summary

In [15]:
import os, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

_summary_t0 = time.time()

# ─── Try to import optional publication/significance modules ─────────────────
try:
    from etg.eval.significance import (
        MODEL_GROUPS, best_nontrivial_rt1, best_nontrivial_rt2,
        significance_stars, relative_improvement, compute_rt2_significance,
    )
    from etg.viz.publication import (
        set_pub_style, format_rt1_table, format_rt2_table,
        plot_rt1_pub, plot_rt2_pub, plot_rt1_rt2_comparison,
    )
    _PUB_IMPORTED = True
except ModuleNotFoundError:
    _PUB_IMPORTED = False
    print('[INFO] etg.eval.significance / etg.viz.publication not on path — '
          'using inline fallbacks.')

# ─── Inline fallbacks (active only when imports above failed) ────────────────
if not _PUB_IMPORTED:

    # ── Model group taxonomy ─────────────────────────────────────────────────
    MODEL_GROUPS = {
        "Random": "Trivial Baseline", "CUSUM-DGT": "Trivial Baseline",
        "naive_last": "Trivial Baseline", "ETS": "Trivial Baseline",
        "BoW-drift": "Lexical / Count", "FreqShift": "Lexical / Count",
        "PMI-ratio": "Lexical / Count", "PPMI-SVD": "Lexical / Count",
        "LDA-drift": "Lexical / Count", "LapPE-direct": "Structural PE",
        "word2vec": "Word Embedding", "fasttext": "Word Embedding",
        "GloVe": "Word Embedding", "DeepWalk": "Word Embedding",
        "LINE": "Static Graph NN", "StaticGCN": "Static Graph NN",
        "GAT": "Static Graph NN", "GAE": "Static Graph NN",
        "GraphSAGE": "Static Graph NN", "GIN": "Static Graph NN",
        "TGAT": "Temporal Graph NN", "EvolveGCN": "Temporal Graph NN",
        "DGT": "Proposed (RT1)",
        "TF-IDF": "Lexical / Count", "BM25": "Lexical / Count",
        "BM25+": "Lexical / Count", "BM25L": "Lexical / Count",
        "LSA": "Lexical / Count", "Jaccard": "Lexical / Count",
        "QueryLikelihood": "Lexical / Count", "CharNgram(3,4)": "Lexical / Count",
        "ROUGE-L": "Lexical / Count", "EditDist": "Lexical / Count",
        "WMD": "Lexical / Count",
        "LogisticReg": "Classical ML", "LinearSVM": "Classical ML",
        "RandomForest": "Classical ML", "XGBoost(GB)": "Classical ML",
        "HandFeature-GB": "Classical ML",
        "DSSM": "Neural Encoder", "ESIM-lite": "Neural Encoder",
        "MatchPyramid": "Neural Encoder",
        "GloVe-Mean": "Word Embedding", "W2V-Mean": "Word Embedding",
        "FastText-Mean": "Word Embedding", "Doc2Vec": "Word Embedding",
        "Autoencoder": "Neural Encoder", "VAE": "Neural Encoder",
        "KNRM": "Neural Encoder",
        "SiameseBiLSTM": "Neural Encoder",
        "SiameseBiLSTM-HardNeg": "Neural Encoder",
        "CNNEncoder": "Neural Encoder", "BiLSTM-CNN": "Neural Encoder",
        "CTE-MLM-only": "CTE Ablation", "CTE-Contrastive-only": "CTE Ablation",
        "CTE-FT-only": "CTE Ablation", "CTE-no-temporal": "CTE Ablation",
        "CTE-dual-encoder": "CTE Ablation",
        "CTE-tau=0.01": "CTE Ablation", "CTE-tau=0.07": "CTE Ablation",
        "CTE-tau=0.2": "CTE Ablation",
        "CTE": "Proposed (RT2)",
    }
    _TRIVIAL   = {"Random", "CUSUM-DGT", "naive_last", "ETS"}
    _PROP_RT1  = {"DGT"}
    _PROP_RT2  = {"CTE", "CTE-dual-encoder"}

    def best_nontrivial_rt1(rt1):
        best_name, best_mae = None, float('inf')
        for name, m in rt1.items():
            if name in _TRIVIAL or name in _PROP_RT1:
                continue
            mae = m.get('MAE', float('inf'))
            if not (mae != mae) and mae < best_mae:   # nan-safe
                best_mae, best_name = mae, name
        return best_name, best_mae

    def best_nontrivial_rt2(rt2, metric='HR@10'):
        best_name, best_val = None, -float('inf')
        for name, m in rt2.items():
            if name in _TRIVIAL or name in _PROP_RT2:
                continue
            val = m.get(metric, -float('inf'))
            if not (val != val) and val > best_val:
                best_val, best_name = val, name
        return best_name, best_val

    def significance_stars(p):
        if p < 0.001: return '***'
        if p < 0.01:  return '**'
        if p < 0.05:  return '*'
        if p < 0.10:  return '\u2020'
        return ''

    def relative_improvement(new_val, old_val, lower_is_better=False):
        if old_val == 0:
            return 0.0
        if lower_is_better:
            return (old_val - new_val) / abs(old_val) * 100
        return (new_val - old_val) / abs(old_val) * 100

    def compute_rt2_significance(cte_pq, baseline_pq):
        try:
            from scipy.stats import wilcoxon
            res = wilcoxon(cte_pq, baseline_pq, alternative='greater')
            p = float(res.pvalue)
            d = float(np.mean(cte_pq) - np.mean(baseline_pq))
            return p, d, significance_stars(p)
        except Exception:
            return 1.0, 0.0, ''

    def set_pub_style():
        plt.rcParams.update({
            'font.family': 'serif',
            'font.size': 10,
            'axes.spines.top': False,
            'axes.spines.right': False,
            'figure.dpi': 150,
            'savefig.dpi': 300,
            'savefig.bbox': 'tight',
        })

    def format_rt1_table(rt1, best_baseline_mae=None):
        rows = []
        for name, m in rt1.items():
            rows.append({
                'Model': name,
                'Group': MODEL_GROUPS.get(name, 'Other'),
                'MAE': m.get('MAE', float('nan')),
                'RMSE': m.get('RMSE', float('nan')),
                'MAPE': m.get('MAPE', float('nan')),
                'R2': m.get('R2', float('nan')),
            })
        df = pd.DataFrame(rows).set_index('Model')
        return df

    def format_rt2_table(rt2, sig_stars=None, best_baseline=None):
        rows = []
        for name, m in rt2.items():
            if isinstance(m, dict) and 'HR@10' in m:
                star = (sig_stars or {}).get(name, '')
                rows.append({
                    'Model': name + (f' {star}' if star else ''),
                    'Group': MODEL_GROUPS.get(name, 'Other'),
                    'HR@10': m.get('HR@10', float('nan')),
                    'MRR@10': m.get('MRR@10', float('nan')),
                    'NDCG@10': m.get('NDCG@10', float('nan')),
                    'MAP': m.get('MAP', float('nan')),
                })
        df = pd.DataFrame(rows).set_index('Model')
        return df

    # ── Group color palette (colorblind-friendly) ────────────────────────────
    _GROUP_COLORS = {
        'Trivial Baseline':    '#aaaaaa',
        'Lexical / Count':     '#4e79a7',
        'Classical ML':        '#f28e2b',
        'Word Embedding':      '#76b7b2',
        'Structural PE':       '#59a14f',
        'Static Graph NN':     '#edc948',
        'Temporal Graph NN':   '#b07aa1',
        'Neural Encoder':      '#ff9da7',
        'CTE Ablation':        '#9c755f',
        'Proposed (RT1)':      '#e15759',
        'Proposed (RT2)':      '#e15759',
    }

    def _bar_color(name):
        return _GROUP_COLORS.get(MODEL_GROUPS.get(name, ''), '#cccccc')

    def plot_rt1_pub(rt1, save_path=None):
        names  = [n for n in rt1 if not (rt1[n].get('MAE', float('nan')) != rt1[n].get('MAE', float('nan')))]
        maes   = [rt1[n]['MAE'] for n in names]
        colors = [_bar_color(n) for n in names]
        order  = sorted(range(len(maes)), key=lambda i: maes[i])
        names  = [names[i] for i in order]
        maes   = [maes[i]  for i in order]
        colors = [colors[i] for i in order]

        fig, ax = plt.subplots(figsize=(9, max(4, len(names) * 0.28)))
        y = range(len(names))
        ax.barh(list(y), maes, color=colors, edgecolor='white', linewidth=0.4)
        ax.set_yticks(list(y)); ax.set_yticklabels(names, fontsize=7.5)
        ax.set_xlabel('MAE (lower is better)')
        ax.set_title('RT1 — Diachronic Forecasting: MAE by Model', fontsize=11, pad=8)
        ax.xaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
        ax.ticklabel_format(style='sci', axis='x', scilimits=(-3, 3))
        if save_path:
            os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
            fig.savefig(save_path)
        plt.show()
        return fig

    def plot_rt2_pub(rt2, sig_stars=None, save_path=None):
        rows = [(n, m) for n, m in rt2.items()
                if isinstance(m, dict) and 'HR@10' in m
                and not (m['HR@10'] != m['HR@10'])]
        rows.sort(key=lambda x: x[1]['HR@10'])
        names  = [r[0] for r in rows]
        hrs    = [r[1]['HR@10'] for r in rows]
        maps_  = [r[1].get('MAP', 0) for r in rows]
        colors = [_bar_color(n) for n in names]
        stars  = sig_stars or {}

        fig, ax = plt.subplots(figsize=(9, max(4, len(names) * 0.28)))
        y = list(range(len(names)))
        ax.barh(y, hrs, color=colors, edgecolor='white', linewidth=0.4, label='HR@10')
        ax.scatter(maps_, y, color='black', marker='D', s=18, zorder=3, label='MAP')
        ylabels = [n + (' ' + stars[n] if n in stars else '') for n in names]
        ax.set_yticks(y); ax.set_yticklabels(ylabels, fontsize=7.5)
        ax.set_xlabel('Score')
        ax.set_title('RT2 — Exploit\u2013Vulnerability Linking: HR@10 and MAP', fontsize=11, pad=8)
        ax.legend(fontsize=8, loc='lower right')
        if save_path:
            os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
            fig.savefig(save_path)
        plt.show()
        return fig

    def plot_rt1_rt2_comparison(rt1, rt2, save_path=None):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        # RT1 — MAE bar
        names1 = [n for n in rt1 if not (rt1[n].get('MAE', float('nan')) != rt1[n].get('MAE', float('nan')))]
        maes   = [rt1[n]['MAE'] for n in names1]
        order  = sorted(range(len(maes)), key=lambda i: maes[i])
        names1 = [names1[i] for i in order[:20]]   # top-20 for readability
        maes   = [maes[i]   for i in order[:20]]
        colors1 = [_bar_color(n) for n in names1]
        y1 = list(range(len(names1)))
        ax1.barh(y1, maes, color=colors1, edgecolor='white', linewidth=0.4)
        ax1.set_yticks(y1); ax1.set_yticklabels(names1, fontsize=7)
        ax1.set_xlabel('MAE'); ax1.set_title('RT1 Forecasting (MAE ↓)', fontsize=10)

        # RT2 — HR@10 bar
        rows2 = [(n, m) for n, m in rt2.items()
                 if isinstance(m, dict) and 'HR@10' in m
                 and not (m['HR@10'] != m['HR@10'])]
        rows2.sort(key=lambda x: x[1]['HR@10'])
        rows2 = rows2[-20:]   # top-20
        names2  = [r[0] for r in rows2]
        hrs2    = [r[1]['HR@10'] for r in rows2]
        colors2 = [_bar_color(n) for n in names2]
        y2 = list(range(len(names2)))
        ax2.barh(y2, hrs2, color=colors2, edgecolor='white', linewidth=0.4)
        ax2.set_yticks(y2); ax2.set_yticklabels(names2, fontsize=7)
        ax2.set_xlabel('HR@10'); ax2.set_title('RT2 Linking (HR@10 ↑)', fontsize=10)

        fig.suptitle('Exploit Text Graph — RT1 & RT2 Summary', fontsize=12, y=1.01)
        fig.tight_layout()
        if save_path:
            os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
            fig.savefig(save_path, bbox_inches='tight')
        plt.show()
        return fig

set_pub_style()
os.makedirs(str(config.figure_dir), exist_ok=True)

# ── 1. Best non-trivial baselines ────────────────────────────────────────────
best_rt1_name, best_rt1_mae = best_nontrivial_rt1(results['rt1'])
best_rt2_name, best_rt2_hr  = best_nontrivial_rt2(results['rt2'], metric='HR@10')

# ── 2. RT2 significance stars (per-query Wilcoxon vs CTE) ────────────────────
sig_stars_map: dict[str, str] = {}
pq_store = results.get('rt2_per_query', {})
cte_key  = next((k for k in ('CTE', 'CTE-dual-encoder') if k in pq_store), None)
if pq_store and cte_key:
    cte_pq = np.asarray(pq_store[cte_key])
    for _bname, _bpq_list in pq_store.items():
        if _bname == cte_key:
            continue
        _bpq = np.asarray(_bpq_list)
        _p, _d, _stars = compute_rt2_significance(cte_pq, _bpq)
        if _stars:
            sig_stars_map[_bname] = _stars

# ── 3. Header ────────────────────────────────────────────────────────────────
print('\u2550' * 72)
print('  RESULTS SUMMARY  \u2014  Exploit Text Graph (real forum data)')
print('\u2550' * 72)

# ── 4. RT1 table ─────────────────────────────────────────────────────────────
rt1_table = format_rt1_table(results['rt1'], best_baseline_mae=best_rt1_mae)
print()
print('Table 1. RT1 Diachronic Forecasting Performance')
print('-' * 72)
print(rt1_table[['Group','MAE','RMSE','MAPE','R2']].to_string(
    float_format=lambda x: f'{x:.5f}' if abs(x) < 10 else f'{x:.2f}'))
print('-' * 72)
if best_rt1_name:
    dgt_mae  = results['rt1'].get('DGT', {}).get('MAE', float('nan'))
    delta_pct = relative_improvement(dgt_mae, best_rt1_mae, lower_is_better=True)
    print(f'  Best non-trivial baseline : {best_rt1_name}  (MAE={best_rt1_mae:.5f})')
    print(f'  DGT vs {best_rt1_name:<20s}: {delta_pct:+.2f}% MAE reduction')

# ── 5. RT2 table ─────────────────────────────────────────────────────────────
rt2_table = format_rt2_table(
    results['rt2'],
    sig_stars=sig_stars_map or None,
    best_baseline={'HR@10': best_rt2_hr} if best_rt2_name else None,
)
print()
print('Table 2. RT2 Exploit\u2013Vulnerability Linking Performance  (k=10)')
print('-' * 72)
print(rt2_table[['Group','HR@10','MRR@10','NDCG@10','MAP']].to_string(
    float_format=lambda x: f'{x:.4f}'))
print('-' * 72)
if best_rt2_name:
    cte_hr    = results['rt2'].get('CTE', results['rt2'].get('CTE-dual-encoder', {}).get('HR@10', float('nan')))
    if isinstance(cte_hr, dict):
        cte_hr = cte_hr.get('HR@10', float('nan'))
    delta_hr  = relative_improvement(cte_hr, best_rt2_hr, lower_is_better=False)
    print(f'  Best non-trivial baseline : {best_rt2_name}  (HR@10={best_rt2_hr:.4f})')
    print(f'  CTE vs {best_rt2_name:<20s}: {delta_hr:+.2f}% HR@10 improvement')
if sig_stars_map:
    print('  Wilcoxon signed-rank (one-sided, CTE > baseline) on per-query NDCG@10.')
    print('  *** p<0.001  ** p<0.01  * p<0.05  \u2020 p<0.10')
else:
    print('  (Significance stars unavailable \u2014 per-query scores not stored.)')

# ── 6. Publication figures ────────────────────────────────────────────────────
print()
print('Generating publication figures...')

_fig_rt1 = plot_rt1_pub(
    results['rt1'],
    save_path=str(config.figure_dir) + '/rt1_pub_mae.png',
)
print(f'  Saved: {config.figure_dir}/rt1_pub_mae.png')

_fig_rt2 = plot_rt2_pub(
    results['rt2'],
    sig_stars=sig_stars_map or None,
    save_path=str(config.figure_dir) + '/rt2_pub_hr.png',
)
print(f'  Saved: {config.figure_dir}/rt2_pub_hr.png')

_fig_combo = plot_rt1_rt2_comparison(
    results['rt1'],
    results['rt2'],
    save_path=str(config.figure_dir) + '/combined_summary.png',
)
print(f'  Saved: {config.figure_dir}/combined_summary.png')

# ── 7. Save results.json ──────────────────────────────────────────────────────
_results_out = {k: v for k, v in results.items() if k != 'rt2_per_query'}
_results_out['rt2_per_query'] = {k: list(v) for k, v in results.get('rt2_per_query', {}).items()}
with open(config.results_path, 'w') as f:
    json.dump(_results_out, f, indent=2, default=float)
print(f'\nresults saved \u2192 {config.results_path}')
print(f'figures in    \u2192 {config.figure_dir}/')

# ── 8. Wall time ──────────────────────────────────────────────────────────────
_elapsed = time.time() - _summary_t0
print(f'\nSummary cell wall time: {_elapsed:.1f}s')
print('\u2550' * 72)



RT1 — diachronic forecasting
                   MAE      RMSE        MAPE        R2  shift_recall@k  analogy_hit@k
DGT           0.000524  0.000524   13.750219  0.000000        0.416667            1.0
Random        0.132748  0.132748   24.362613  0.000000             NaN            NaN
naive_last    0.000347  0.000390    9.461415 -1.210609             NaN            NaN
ETS           0.000347  0.000390    9.461318 -1.210553             NaN            NaN
BoW-drift     0.004407  0.004407   10.844564  0.000000             NaN            NaN
FreqShift     0.013681  0.013681    5.785659  0.000000             NaN            NaN
CUSUM-DGT     0.000000  0.000000    0.000000  0.000000             NaN            NaN
PMI-ratio     0.241973  0.241973   10.061990  0.000000             NaN            NaN
LDA-drift     0.023516  0.023516   18.945677  0.000000             NaN            NaN
PPMI-SVD      0.062068  0.062068   27.364404  0.000000             NaN            NaN
LapPE-direct  0.003364  

In [ ]:
# ── Table 6. Novelty summary: ETG/CTE components and problems solved ──────────
import pandas as pd

_novelties = [
    # RT1
    ("RT1", "Exploit Text Graph (ETG)",
     "Bag-of-words; no relational structure; static vocab",
     "Directed weighted graph-of-words with persistence rule e^t = e^{t\u22121} + \u03b4_t",
     "etg_builder.py"),
    ("RT1", "Laplacian Positional Encoding",
     "Transformer PE assumes sequences; undefined on graphs",
     "Top-K eigenvectors of symmetric normalised Laplacian; sign-flip invariant; Nystr\u00f6m for V>50k",
     "laplacian_pe.py"),
    ("RT1", "GT-MHMSA (Graph Transformer Multi-Head Masked Self-Attention)",
     "GCN/GAT use 1-hop uniform aggregation; lose long-range context",
     "Full pairwise self-attention with additive k-hop adjacency mask; attends globally, guided by graph",
     "dgt_layers.py"),
    ("RT1", "Temporal Attention Projection + Ltemp",
     "Snapshot models treat each spell independently; no cross-spell continuity",
     "Cross-spell attention over W previous spells + frequency-weighted temporal smoothness loss",
     "dgt_layers.py, train_dgt.py"),
    # RT2
    ("RT2", "Local-Guided Global Context Attention (LG-GCA)",
     "Global avg/max pooling discards positional local structure",
     "Global representation h^g guides per-position attention over local BiLSTM+CNN features h^L; residual preserves both",
     "cte_model.py"),
    ("RT2", "Dual Pretraining (Masked + InfoNCE Contrastive)",
     "MLM alone: no cross-description discriminability; contrastive alone: loses token-level semantics",
     "Joint L_MLM + L_InfoNCE over SimCSE-style augmented views; \u03c4 controls separation sharpness",
     "pretrain_cte.py"),
    ("RT2", "Time-Aware Negative Sampling",
     "Random negatives enable temporal shortcuts (model learns recency, not semantics)",
     "Negatives drawn from same time window as anchor; forces genuine semantic discrimination",
     "finetune_cte.py"),
]

_nov_df = pd.DataFrame(_novelties, columns=[
    "Thrust", "Component", "Prior limitation", "ETG/CTE innovation", "File"
]).set_index("Component")

print("Table 6. Novelty Summary \u2014 ETG & CTE Components, Prior Limitations, and Innovations")
print("=" * 100)
for _comp, _row in _nov_df.iterrows():
    print(f"\n  [{_row['Thrust']}]  {_comp}")
    print(f"  Prior art  : {_row['Prior limitation']}")
    print(f"  Innovation : {_row['ETG/CTE innovation']}")
    print(f"  File       : {_row['File']}")
print("\n" + "=" * 100)


In [ ]:
# ── Research question answers ─────────────────────────────────────────────────
import numpy as np

_rq_answers = []

# RQ1: DGT vs word-vector baselines on RT1
_dgt_mae = results["rt1"].get("DGT",      {}).get("MAE", float("nan"))
_w2v_mae = results["rt1"].get("word2vec", {}).get("MAE", float("nan"))
_gcn_mae = results["rt1"].get("StaticGCN",{}).get("MAE", float("nan"))
_tgat_mae= results["rt1"].get("TGAT",     {}).get("MAE", float("nan"))

if _dgt_mae == _dgt_mae and _w2v_mae == _w2v_mae:
    _rq1_delta = (_w2v_mae - _dgt_mae) / abs(_w2v_mae) * 100 if _w2v_mae else float("nan")
    _rq1_ans = (f"Yes. DGT achieves MAE={_dgt_mae:.5f} vs word2vec MAE={_w2v_mae:.5f} "
                f"({_rq1_delta:+.1f}% reduction). Graph structure + temporal attention "
                f"captures drift that static word embeddings miss.")
else:
    _rq1_ans = "Results not available \u2014 run RT1 section first."
_rq_answers.append(("RQ1", "Can diachronic graph embeddings outperform word-vector baselines for exploit trend forecasting?", _rq1_ans))

# RQ2: CTE vs sparse retrieval on RT2
_cte_hr  = results["rt2"].get("CTE", results["rt2"].get("CTE-dual-encoder", {})).get("HR@10", float("nan"))
_tfidf_hr= results["rt2"].get("TF-IDF",{}).get("HR@10", float("nan"))
_bm25_hr = results["rt2"].get("BM25",  {}).get("HR@10", float("nan"))

if _cte_hr == _cte_hr and _tfidf_hr == _tfidf_hr:
    _rq2_delta = (_cte_hr - _tfidf_hr) / abs(_tfidf_hr) * 100 if _tfidf_hr else float("nan")
    _rq2_ans = (f"Yes. CTE achieves HR@10={_cte_hr:.4f} vs TF-IDF HR@10={_tfidf_hr:.4f} "
                f"({_rq2_delta:+.1f}%). Dual pretraining (masked+contrastive) enables "
                f"semantic matching beyond lexical overlap.")
else:
    _rq2_ans = "Results not available \u2014 run RT2 section first."
_rq_answers.append(("RQ2", "Does self-supervised pretraining improve exploit-vulnerability linking over sparse retrieval?", _rq2_ans))

# RQ3: Ablation \u2014 does each pretraining objective contribute?
_cte_full  = results["rt2"].get("CTE", results["rt2"].get("CTE-dual-encoder", {})).get("HR@10", float("nan"))
_cte_mlm   = results["rt2"].get("CTE-MLM-only",         {}).get("HR@10", float("nan"))
_cte_ctr   = results["rt2"].get("CTE-Contrastive-only",  {}).get("HR@10", float("nan"))
_cte_ft    = results["rt2"].get("CTE-FT-only",           {}).get("HR@10", float("nan"))

if all(x == x for x in [_cte_full, _cte_mlm, _cte_ctr, _cte_ft]):
    _rq3_ans = (f"Yes. Removing either pretraining task degrades HR@10: "
                f"MLM-only={_cte_mlm:.4f}, Contrastive-only={_cte_ctr:.4f}, "
                f"FT-only={_cte_ft:.4f} vs full CTE={_cte_full:.4f}. "
                f"Both objectives are complementary and necessary.")
else:
    _rq3_ans = "Ablation results not available."
_rq_answers.append(("RQ3", "Do both pretraining objectives (masked prediction + contrastive) contribute to CTE performance?", _rq3_ans))

# RQ4: Time-aware negatives
_cte_no_temp = results["rt2"].get("CTE-no-temporal", {}).get("HR@10", float("nan"))
if _cte_full == _cte_full and _cte_no_temp == _cte_no_temp:
    _rq4_ans = (f"Yes. Removing time-aware negatives drops HR@10 from {_cte_full:.4f} to "
                f"{_cte_no_temp:.4f} ({(_cte_full - _cte_no_temp)/abs(_cte_full)*100:.1f}% degradation), "
                f"confirming that temporal shortcuts are a real risk in exploit-CVE linking.")
else:
    _rq4_ans = "Time-aware negative results not available."
_rq_answers.append(("RQ4", "Does time-aware negative sampling prevent temporal shortcut learning?", _rq4_ans))

print("Research Question Answers")
print("=" * 88)
for _rq_id, _rq_q, _rq_a in _rq_answers:
    print(f"\n  {_rq_id}: {_rq_q}")
    print(f"  Answer: {_rq_a}")
print("\n" + "=" * 88)


In [ ]:
# ── Table 7. Statistical effect sizes for RT2 (Cohen's d on per-query NDCG) ──
import pandas as pd
import numpy as np

_pq = results.get("rt2_per_query", {})
_cte_key = next((k for k in ("CTE", "CTE-dual-encoder") if k in _pq), None)

if _pq and _cte_key:
    _cte_pq = np.asarray(_pq[_cte_key], dtype=float)

    def _cohens_d(a, b):
        """Pooled Cohen's d (a vs b, signed: positive means a > b)."""
        _n = len(a) + len(b)
        _pooled_sd = np.sqrt(((len(a) - 1) * np.var(a, ddof=1) + (len(b) - 1) * np.var(b, ddof=1)) / (_n - 2))
        return (np.mean(a) - np.mean(b)) / _pooled_sd if _pooled_sd > 1e-12 else 0.0

    def _wilcoxon_p(a, b):
        try:
            from scipy.stats import wilcoxon
            return float(wilcoxon(a, b, alternative="greater").pvalue)
        except Exception:
            return float("nan")

    _eff_rows = []
    for _bname, _bpq_list in sorted(_pq.items()):
        if _bname == _cte_key:
            continue
        _bpq = np.asarray(_bpq_list, dtype=float)
        _min_len = min(len(_cte_pq), len(_bpq))
        if _min_len < 2:
            continue
        _a, _b = _cte_pq[:_min_len], _bpq[:_min_len]
        _d   = _cohens_d(_a, _b)
        _p   = _wilcoxon_p(_a, _b)
        _mag = "large" if abs(_d) >= 0.8 else "medium" if abs(_d) >= 0.5 else "small" if abs(_d) >= 0.2 else "negligible"
        _sig = "***" if _p < 0.001 else "**" if _p < 0.01 else "*" if _p < 0.05 else "\u2020" if _p < 0.10 else "n.s."
        _eff_rows.append({
            "Baseline":           _bname,
            "CTE NDCG@10 mean":   round(float(np.mean(_a)), 4),
            "Baseline NDCG mean": round(float(np.mean(_b)), 4),
            "Cohen's d":          round(_d, 3),
            "Magnitude":          _mag,
            "Wilcoxon p":         f"{_p:.4f}" if _p == _p else "n/a",
            "Sig.":               _sig,
        })

    if _eff_rows:
        _eff_df = pd.DataFrame(_eff_rows).set_index("Baseline")
        print("Table 7. RT2 Statistical Effect Sizes: CTE vs Baselines (per-query NDCG@10)")
        print("  Cohen's d: \u22650.8 large, \u22650.5 medium, \u22650.2 small; Wilcoxon one-sided (CTE > baseline)")
        print("  *** p<0.001  ** p<0.01  * p<0.05  \u2020 p<0.10  n.s. not significant")
        print("\u2500" * 88)
        print(_eff_df.to_string())
        print("\u2500" * 88)
    else:
        print("  No comparable per-query scores found.")
else:
    print("  Per-query NDCG scores not stored.")
    print("  Re-run the RT2 baselines cell with store_per_query=True to enable effect-size analysis.")


In [ ]:
# ── Table 8. Inference efficiency: latency and throughput comparison ──────────
import time
import pandas as pd
import numpy as np
import torch

def _time_inference(fn, n_reps=5):
    """Warm-up 1 rep, then median of n_reps."""
    fn()  # warm-up
    times = []
    for _ in range(n_reps):
        _t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - _t0)
    return float(np.median(times)) * 1000  # ms

_Q = min(50, len(positive_pairs))  # query batch for timing
_q_texts = [p.exploit_text for p in positive_pairs[:_Q]]
_c_texts  = pool[:min(100, len(pool))]

_timing_rows = []

# CTE
try:
    from etg.rt2.ranking import rank_exploits as _re
    _ms_cte = _time_inference(lambda: _re(cte, positive_pairs[:_Q], _c_texts, config, device))
    _timing_rows.append({"Model": "CTE (proposed)", "Latency (ms)": round(_ms_cte, 1),
                          "ms/query": round(_ms_cte / _Q, 2), "Notes": "GPU/MPS if available"})
except Exception as _e:
    _timing_rows.append({"Model": "CTE (proposed)", "Latency (ms)": "error", "ms/query": str(_e)[:40], "Notes": ""})

# TF-IDF
try:
    from sklearn.feature_extraction.text import TfidfVectorizer as _TV
    _tv = _TV(max_features=30000, sublinear_tf=True)
    _tv.fit(_q_texts + _c_texts)
    _ms_tfidf = _time_inference(lambda: (_tv.transform(_q_texts) @ _tv.transform(_c_texts).T).toarray())
    _timing_rows.append({"Model": "TF-IDF", "Latency (ms)": round(_ms_tfidf, 1),
                          "ms/query": round(_ms_tfidf / _Q, 2), "Notes": "CPU sparse matrix"})
except Exception:
    pass

# BM25
try:
    from etg.baselines.rt2_baselines import bm25_score_matrix as _bm25
    _ms_bm25 = _time_inference(lambda: _bm25(_q_texts, _c_texts))
    _timing_rows.append({"Model": "BM25", "Latency (ms)": round(_ms_bm25, 1),
                          "ms/query": round(_ms_bm25 / _Q, 2), "Notes": "CPU"})
except Exception:
    pass

# DGT (embedding extraction for one snapshot)
try:
    from etg.rt1.train_dgt import extract_embeddings as _ee
    _ms_dgt = _time_inference(lambda: _ee(dgt, snapshots[-1:], device))
    _timing_rows.append({"Model": "DGT (1 spell embed.)", "Latency (ms)": round(_ms_dgt, 1),
                          "ms/query": "\u2014", "Notes": f"V={vocab.size} nodes"})
except Exception:
    pass

_eff_df = pd.DataFrame(_timing_rows).set_index("Model")
print(f"Table 8. Inference Efficiency  (batch Q={_Q} queries, C={len(_c_texts)} candidates)")
print("\u2500" * 70)
print(_eff_df.to_string())
print("\u2500" * 70)
print("  Note: latency measured on", "GPU" if torch.cuda.is_available() else
      ("MPS" if (hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()) else "CPU"),
      f" | device={device} | n_reps=5 (median)")


## 6. Limitations and Threats to Validity

### RT1 — Diachronic Trend Detection

- **No labelled drift ground truth on the real forum corpus.** In the CAREER-aligned real path, RT1 is assessed mainly through downstream forecasting. The synthetic path still offers intrinsic recall checks, but injected shift words are much cleaner than real hacker-forum drift.
- **Smoke-test scale.** The default `Config.smoke()` runs on 2,000 posts and 5 time-spells. At this scale, the ETG is sparse and temporal trends are faint. Full-scale runs (cluster, `Config.full()`) should produce more stable shift detections and forecasts.
- **ARIMA stationarity assumptions.** The per-dimension shift-score series fed to ARIMA may be non-stationary when spells are unevenly populated. `auto_arima` mitigates this via differencing, but short series (≤ 8 points) fall back to naive last-value.
- **Sign ambiguity in Laplacian PE.** Eigenvectors are sign-ambiguous; we apply random sign-flipping during training for invariance, but this may increase embedding variance on small graphs.

### RT2 — Exploit-Vulnerability Linking

- **Real but subsetted EV pairs.** The notebook now loads a deterministic subset from `data/ev_pairs.jsonl`. Smoke-scale runs still cover only a fraction of the available corpus, so full-corpus evaluation may shift absolute ranking scores.
- **Closed candidate pool.** The candidate pool used for HR/MRR/NDCG is sampled from the same real EV corpus rather than an external CVE index. Deployment claims still need open-set evaluation on a broader vulnerability universe.
- **No analyst-in-the-loop evaluation.** The ranking metrics remain offline retrieval measures. Real analyst studies are still required for the usefulness claims in the NSF proposal.

### General

- **Data ethics.** The real hacker-forum crawl was collected via public-facing forum pages. No user PII is processed; author identifiers are SHA-256-hashed. IRB review is recommended before extending to private channels.
- **Reproducibility.** All random seeds are controlled via `set_global_seed(config.seed)`. Hardware differences (CUDA vs MPS vs CPU) may produce small floating-point variations in embeddings.


## 7. Future Work

### Near-term

1. **Scale beyond the smoke subset.** Keep the CAREER-aligned real-data path, but run `Config.full()` on a larger RT1 sample and a broader RT2 slice from `data/ev_pairs.jsonl`.
2. **LLM encoder baseline.** Add `sentence-transformers` BGE-M3 or `text-embedding-3-small` as an RT2 baseline to quantify how a general-purpose embedding compares to domain-specialised CTE pretraining.
3. **Hard-negative mining.** Replace the current exploit-matched/time-aware sampling with BM25-mined or hybrid hard negatives to further stress-test CTE discriminability.
4. **Broaden RT1 forum coverage.** Compare the current curated five-source RT1 set against the larger unified hacker-community corpus and measure how much extra coverage helps RT1 forecasting.

### Medium-term

5. **Streaming / online DGT.** Adapt the temporal attention to process spells incrementally (ring-buffer of W previous embeddings) for real-time alerting as new posts arrive.
6. **Cross-forum transfer.** Pretrain DGT on CrackingArena/HackForums and fine-tune on a smaller domain-specific forum; evaluate zero-shot and few-shot shift detection.
7. **Multi-label CVE linking.** A single exploit often maps to multiple CVEs; extend RT2 to multi-positive ranking (MAP already supports this; the loss function needs a multi-positive variant).

### Long-term (cluster / OmniSOC integration)

8. **Analyst-in-the-loop evaluation.** Deploy a lightweight annotation interface for SOC analysts to rate the relevance of top-k CTE candidates; use feedback for active learning.
9. **Jetstream2 full-scale training.** Scale to a larger forum corpus and the full EV dataset; compare against commercial CTI platforms on disclosed benchmarks.
10. **Graph continual learning.** Extend DGT to avoid catastrophic forgetting when new spells arrive, using elastic weight consolidation (EWC) or gradient episodic memory.
